In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import fitsio
from pycorr import TwoPointCorrelationFunction, TwoPointEstimator, project_to_multipoles, project_to_wp, utils, setup_logging
from scipy.optimize import curve_fit
from LSS.common_tools import mknz, dl
from astropy.table import Table
import itertools

from dataloc import *
from groupcatalog import read_wp_file
from pyutils import get_max_observable_z
from desiclusteringtools import *

import plotting as pp

def savefig(filename, **kwargs):
    """Save the current figure as a PNG file and close it."""
    path = os.path.join(PAPER_PLOT_FOLDER, f'{filename}.png')
    plt.savefig(path, bbox_inches='tight', **kwargs)

# MAKE ALL PLOTS TEXT BIGGER
plt.rcParams.update({'font.size': 15})
# But legend a bit smaller
plt.rcParams.update({'legend.fontsize': 12})
# Set DPI up a bit
plt.rcParams['figure.dpi'] = pp.DPI_PAPER

In [ ]:
# First, figure out what BGS mag limit is equiv to SDSS maglimit due to photometry differences.
maglims = [-22.405, -21.815, -21.21, -20.64, -19.52] # These are the adjusted mag limits to match the number density of SDSS
zmaxes = [0.245169, 0.198666, 0.158833, 0.132333, 0.084833] # Fix volume to match SDSS
zmins = [0.02, 0.02, 0.02, 0.02, 0.02]
bgs_footprint = 7472 
sdss_footprint = 7700 # SDSS Footprint in Zehavi 2011 is 7700 deg^2
sdss_mags = [-22.0, -21.5, -21.0, -20.5, -19.5] # SDSS Zehavi 2011 magnitude thresholds
sdss_counts = [11385, 39456, 83238, 132225, 132664] # Number of galaxies in Zehavi 2011 for each threshold sample


# Determine Mag Limits to Match ndens

In [ ]:
# I don't know SDSS's exact n(z) so let's just do the overall density.
def galdensity(zmin, zmax, footprint, galcount):
    # Use DESI's dl method for luminosity distance to get the physical volume between zmin and zmax, then divide galcount by that volume to get density
    vol = (footprint/41253) * 4/3 * np.pi * (dl(zmax)**3 - dl(zmin)**3)
    return galcount / vol

# Function to integrate the two across a z range
def galdensity_nz(zmin, zmax):
    # Find idx for zmin and zmax
    idx1 = np.argmin(np.abs(NGC_nz[:,0] - zmin))
    idx2 = np.argmin(np.abs(NGC_nz[:,0] - zmax))

    # NGC
    zwidth = NGC_nz[:,2] - NGC_nz[:,1]
    integrated = NGC_nz[:,3] * zwidth 
    summed = integrated[idx1:idx2].sum()

    # And now for SGC
    idx1 = np.argmin(np.abs(SGC_nz[:,0] - zmin))
    idx2 = np.argmin(np.abs(SGC_nz[:,0] - zmax))
    zwidth = SGC_nz[:,2] - SGC_nz[:,1]
    integrated = SGC_nz[:,3] * zwidth
    summed += integrated[idx1:idx2].sum()

    return summed

# Make a luminosity threshold sample like SDSS to see if n(z) is same
def make_sdsslike_cuts(fcd, maglim, zmax, zmin):
    writename = fcd.replace('.dat.fits', f'_testcut.dat.fits')  
    arr = fitsio.read(fcd)
    arr = arr[arr['Z'] < zmax]
    arr = arr[arr['Z'] > zmin]
    arr = arr[arr['ABSMAG_R'] < maglim] # I *think* this is ABSMAG01_SDSS_R, my prep-extra-cols.py says so...
    fitsio.write(writename, arr, clobber=True)

# Function to combine the two to get an overall density, appropiatly weighting each by volume
def combine_nz(nz1, nz2):
    combined = np.zeros((len(nz1), 6))
    combined[:,0] = nz1[:,0]
    combined[:,1] = nz1[:,1]
    combined[:,2] = nz1[:,2]
    combined[:,4] = nz1[:,4] + nz2[:,4]
    combined[:,5] = nz1[:,5] + nz2[:,5]
    combined[:,3] = combined[:,4] / combined[:,5]
    return combined

for maglim, zmax, zmin, sdss_maglim, sdss_count in zip(maglims, zmaxes, zmins, sdss_mags, sdss_counts):
    #if zmax != 0.132333:
    #    continue
    outpath = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_FOOTPRINT.txt'
    outpath2 = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_FOOTPRINT.txt'

    make_sdsslike_cuts('/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_clustering.dat.fits', maglim, zmax, zmin)
    make_sdsslike_cuts('/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_clustering.dat.fits', maglim, zmax, zmin)

    cat = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_clustering_testcut.dat.fits'
    ran = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_0_clustering.ran.fits'
    mknz(cat, ran, outpath, zmax=0.3)

    cat = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_clustering_testcut.dat.fits'
    ran = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_0_clustering.ran.fits'
    mknz(cat, ran, outpath2, zmax=0.3)

    # zmid zlow zhigh n(z) Nbin Vol_bin
    NGC_nz = np.loadtxt(outpath)
    SGC_nz = np.loadtxt(outpath2)

    total_nz = combine_nz(NGC_nz, SGC_nz)
    galcount = total_nz[:,4].sum()
    bgs_density = galdensity(0.02, zmax, bgs_footprint,  galcount)
    sdss_density = galdensity(0.02, zmax, sdss_footprint, sdss_count)

    print(f"BGS BRIGHT Y1 number of galaxies with M_r < {maglim} and z < {zmax}: {galcount:,.0f}. Density = {bgs_density :.4e} Mpc^-3")
    print(f"SDSS Zehavi 2011 number of galaxies with M_r < {sdss_maglim} and z < {zmax} :  {sdss_count:,}. Density = {sdss_density :.4e} Mpc^-3")
    print(f"Difference (%): {(bgs_density - sdss_density) / sdss_density * 100 :.2f}%")


In [ ]:
# Print off what the zmax should be if you want to use the 19.5 BGS flux limit for those magnitudes.
for maglim in maglims:
    zmax = get_max_observable_z(maglim, 19.5)
    print(f"For M_r < {maglim}, the zmax corresponding to the BGS flux limit of r=19.5 is {zmax:.4f}")

# View Results

In [ ]:
clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/newclustering/Y1/LSS/iron/LSScats/v1.5pip/SDSS_COMPARE/'

In [ ]:
# Find all *_linclusimsysfit.png files in the clustering_base_dir and its subdirectories
import glob
linclusimsysfit_files = glob.glob(os.path.join(clustering_base_dir, '**', '*_linclusimsysfit.png'), recursive=True) 
print(f"Found {len(linclusimsysfit_files)} linclusimsysfit.png files:")
for f in linclusimsysfit_files:
    print(f" - {f}")

# First linclusimsysfit_files investigation
for i in range(10):
    first_file = linclusimsysfit_files[i]
    plt.figure(figsize=(10, 10))
    img = plt.imread(first_file)
    plt.imshow(img)
    plt.axis('off') 
    plt.show()

In [ ]:
all_results = load_allcounts_from_disk(clustering_base_dir)

if all_results:
    print("\nExample of first loaded result:")
    first_result = all_results[0]
    print("Parameters:", first_result['params'])
    print("Data object:", first_result['data'])


In [ ]:
zehavi_bins = np.logspace(np.log10(0.13), np.log10(33), 13)
with np.printoptions(precision=3, suppress=True): 
    print(zehavi_bins)
    # Print the centers of these bins now
    print((zehavi_bins[:-1] + zehavi_bins[1:]) / 2) 
    # This is close to what I ripped off of the paper, so let's use this for plotting them for consistency.

In [ ]:
from scipy.stats import chi2

def chi2_compare_bgs_versions(all_results, weight_type):
    """
    For each magnitude threshold, compare wp measured with SDSS zmaxes vs BGS zmaxes
    using a chi-squared test, ignoring the x-axis offset.
    """
    filtered = [
        r for r in all_results
        if r['params']['weights'] == weight_type
        and r['params']['mag_thresh'] is not None
        and r['params']['sample_type'] == 'ALL'
    ]

    # Group by thresh, then by zmax type
    from collections import defaultdict
    by_thresh = defaultdict(list)
    for r in filtered:
        by_thresh[r['params']['mag_thresh']].append(r)

    for thresh, results in sorted(by_thresh.items(), key=lambda x: float(x[0])):
        if len(results) != 2:
            print(f"M_r < -{thresh}: expected 2 results (SDSS zmax vs BGS zmax), got {len(results)}, skipping.")
            continue

        # Sort by zmax so r[0] = smaller (SDSS) zmax, r[1] = larger (BGS) zmax
        results = sorted(results, key=lambda r: float(r['params']['zmax']))
        r_sdss, r_bgs = results

        s_sdss: TwoPointEstimator = r_sdss['data']
        s_bgs: TwoPointEstimator  = r_bgs['data']

        rp_sdss, wp_sdss, cov_sdss = s_sdss.get_corr(return_sep=True, return_cov=True, mode='wp')
        rp_sdss = rp_sdss[1:]  # Skip the first bin which is likely affected by fiber collisions
        wp_sdss = wp_sdss[1:]
        cov_sdss = cov_sdss[1:, 1:]
        rp_bgs,  wp_bgs,  cov_bgs  = s_bgs.get_corr(return_sep=True, return_cov=True, mode='wp')
        rp_bgs = rp_bgs[0:9]  # Skip the first bin which is likely affected by fiber collisions
        wp_bgs = wp_bgs[0:9]
        cov_bgs = cov_bgs[0:9, 0:9]

        # Use diagonal errors only (ignore covariance between bins)
        err_sdss = np.sqrt(np.diag(cov_sdss))
        err_bgs  = np.sqrt(np.diag(cov_bgs))

        # Combined error in quadrature
        err_combined = np.sqrt(err_sdss**2 + err_bgs**2)

        # Assume same number of bins; ignore x offset
        n = min(len(wp_sdss), len(wp_bgs))
        diff = wp_sdss[:n] - wp_bgs[:n]
        chi2_val = np.sum((diff / err_combined[:n])**2)
        dof = n
        p_val = 1 - chi2.cdf(chi2_val, dof)

        print(f"M_r < -{thresh}:  zmax_SDSS={r_sdss['params']['zmax']}, zmax_BGS={r_bgs['params']['zmax']}")
        print(f"  chi2 = {chi2_val:.2f}, dof = {dof}, chi2/dof = {chi2_val/dof:.2f}, p = {p_val:.3f}")
        print()

chi2_compare_bgs_versions(all_results, 'pip_angular_bitwise')
# According to this, there is possibly a statistically significant difference only in the < -21.21 sample between the two zmaxes.

In [ ]:
wp_thresholds(all_results, 'pip_angular_bitwise')

In [ ]:
def chi2_compare_bgs_to_sdss(all_results, weight_type, sdss_data):
    """
    Compare BGS wp (using BGS zmaxes) to actual SDSS Zehavi+11 data,
    using only BGS error bars as the denominator (since SDSS errors unavailable).
    
    sdss_data: dict mapping sdss mag threshold (float) -> (rp array, wp array)
               e.g. {22.0: (rp, wp), 21.5: (rp, wp), ...}
    """
    from scipy.stats import chi2 as chi2_dist

    # Map BGS mag thresholds to SDSS ones (from your notebook)
    bgs_to_sdss_mag = {
        '22.405': -22.0,
        '21.815': -21.5,
        '21.21':  -21.0,
        '20.64':  -20.5,
        '19.52':  -19.5,
    }

    filtered = [
        r for r in all_results
        if r['params']['weights'] == weight_type
        and r['params']['mag_thresh'] is not None
        and r['params']['sample_type'] == 'ALL'
    ]

    from collections import defaultdict
    by_thresh = defaultdict(list)
    for r in filtered:
        by_thresh[r['params']['mag_thresh']].append(r)

    for thresh, results in sorted(by_thresh.items(), key=lambda x: float(x[0])):
        sdss_mag = bgs_to_sdss_mag.get(thresh)
        if sdss_mag is None or sdss_mag not in sdss_data:
            print(f"M_r < -{thresh}: no matching SDSS data, skipping.")
            continue

        # Pick the BGS result with the BGS zmax (largest zmax)
        r_bgs = max(results, key=lambda r: float(r['params']['zmax']))
        s_bgs: TwoPointEstimator = r_bgs['data']

        rp_bgs, wp_bgs, cov_bgs = s_bgs.get_corr(return_sep=True, return_cov=True, mode='wp')
        err_bgs = np.sqrt(np.diag(cov_bgs))

        rp_sdss, wp_sdss = sdss_data[sdss_mag]

        # Interpolate SDSS onto BGS rp points (or vice versa)
        wp_sdss_interp = np.interp(rp_bgs, rp_sdss, wp_sdss)

        diff = wp_bgs - wp_sdss_interp
        chi2_val = np.sum((diff / (3 * err_bgs))**2)
        dof = len(wp_bgs)
        p_val = 1 - chi2_dist.cdf(chi2_val, dof)

        print(f"M_r < -{thresh} (SDSS M_r < -{sdss_mag}):  zmax_BGS={r_bgs['params']['zmax']}")
        print(f"  chi2 = {chi2_val:.2f}, dof = {dof}, chi2/dof = {chi2_val/dof:.2f}, p = {p_val:.5f}")
        print(f"  NOTE: chi2 uses only BGS errors; underestimates true uncertainty.")
        print()

# Usage - load your ripped SDSS points into this dict first:
sdss_data = {}

from dataloc import PARAMS_SDSS_FOLDER
zehavi_bins = np.logspace(np.log10(0.13), np.log10(33), 13)

for m in sdss_mags:
    savedir = PARAMS_SDSS_FOLDER + f'sdss-thresh{m:.1f}.csv'
    if not os.path.exists(savedir):
        print(f'File {savedir} not found')
        continue
    data = np.loadtxt(savedir, skiprows=1, dtype='float', delimiter=',')
    x = (zehavi_bins[:-1] + zehavi_bins[1:]) / 2
    print(np.shape(data))
    if np.shape(data)[0] == 11:
        x = x[1:]  # Skip first point for the faintest bin since it's not in the SDSS data I ripped
    sdss_data[m] = (x, data[:,1])

chi2_compare_bgs_to_sdss(all_results, 'pip_angular_bitwise', sdss_data)

In [ ]:
compare_wp_thresholds_to_sdss(all_results, 'pip_angular_bitwise')
savefig('wp_thresholds_sdss_comparison')
plt.show()